In [1]:
from collections import Counter, defaultdict
import json
import pandas as pd
from pathlib import Path

In [84]:
# dataset_path = Path("/Users/fangzhouma/Desktop/3d_vision/3D-VLM-benchmark-OOS/outputs/jsonl_v2/multi_turn.jsonl")
dataset_path = Path("/Users/fangzhouma/Desktop/3d_vision/3D-VLM-benchmark-OOS/outputs/jsonl_v2/multi_turn.jsonl")
# Change this path if your JSONL is elsewhere.

raw_docs = []
with dataset_path.open("r") as f:
    for line in f:
        raw_docs.append(json.loads(line))

len(raw_docs), raw_docs[0].keys()

(100,
 dict_keys(['trajectory_id', 'question_class', 'video_id', 'video_path', 'query_time_sec', 'query_time_in_clip_sec', 'clip_start_time_sec', 'clip_end_time_sec', 'clip_duration_sec', 'horizon_sec', 'object_a_assoc_id', 'object_a_name', 'num_incremental_steps', 'num_branch_steps', 'terminated_at_step', 'stop_reason', 'generation_info', 'doc_id', 'mode', 'steps', 'include_gold_history', 'gold_history_messages']))

In [85]:
rows = []

for raw in raw_docs:
    for step in raw.get("steps", []):
        if step.get("skipped"):
            continue

        rows.append({
            "trajectory_id": raw.get("trajectory_id"),
            "video_id": raw.get("video_id"),
            "doc_id": raw.get("doc_id"),
            "mode": raw.get("mode"),
            "query_time_sec": raw.get("query_time_sec"),
            "step": str(step.get("step")),
            "branch_group": step.get("branch_group"),
            "depends_on_steps": step.get("depends_on_steps"),
            "question_class": step.get("step_question_class"),
            "question": step.get("question"),
            "choices": step.get("choices") or [],
            "correct_idx": step.get("correct_idx"),
            "target_text": step.get("target_text"),
            "acceptable_idxs": step.get("acceptable_idxs"),
            "answer_metadata": step.get("answer_metadata"),
        })

df = pd.DataFrame(rows)
df.head()

,trajectory_id,video_id,doc_id,mode,query_time_sec,step,branch_group,depends_on_steps,question_class,question,choices,correct_idx,target_text,acceptable_idxs,answer_metadata
0,P01-20240203-132119__oos_staged_h5p0_142,P01-20240203-132119,P01-20240203-132119__oos_staged_h5p0_142,multi_turn,794.0,1,NaN,[],oos_step1_visibility,"At the current time <TIME 00:13:14.0 video 1>,...","[No, Yes]",0.0,No,None,"{'status': 'out_of_view', 'is_visible': False,..."
1,P01-20240203-132119__oos_staged_h5p0_142,P01-20240203-132119,P01-20240203-132119__oos_staged_h5p0_142,multi_turn,794.0,2,NaN,[1],oos_step2_last_visible,The pen was moved earlier in the video. When w...,[],NaN,"<TIME 00:13:08.0 video 1>; Point=(0.7052, 0.9480)",None,"{'sampled_last_visible_time_sec': 788.0, 'samp..."
2,P01-20240203-132119__oos_staged_h5p0_142,P01-20240203-132119,P01-20240203-132119__oos_staged_h5p0_142,multi_turn,794.0,3,NaN,"[1, 2]",oos_step3_last_placement,The pen was moved earlier in the video. At wha...,[],NaN,"<TIME 00:01:23.4 video 1>; Point=(0.6961, 0.8206)",None,"{'last_placement_time_sec': 83.43333333333334,..."
3,P01-20240203-132119__oos_staged_h5p0_142,P01-20240203-132119,P01-20240203-132119__oos_staged_h5p0_142,multi_turn,794.0,4,NaN,"[1, 2, 3]",oos_step4_fixture,"At the current time <TIME 00:13:14.0 video 1>,...","[shelf, counter area next to the window, dishw...",3.0,counter area close to the microwave,None,"{'reference_time_sec': 83.43333333333334, 'cor..."
4,P01-20240203-132119__oos_staged_h5p0_142,P01-20240203-132119,P01-20240203-132119__oos_staged_h5p0_142,multi_turn,794.0,5a,post_step4,"[1, 2, 3, 4]",oos_branch_object_camera_relative_position,"At the current time <TIME 00:13:14.0 video 1>,...","[Front-left, Back-right, Back-left, Front-right]",3.0,Front-right,None,"{'reference_time_sec': 794.0, 'camera_coordina..."


In [86]:
for qclass, g in df.groupby("question_class"):
    print("\n" + "=" * 100)
    print(qclass)
    print("N:", len(g))

    print("\nTarget text distribution:")
    print(g["target_text"].value_counts(dropna=False))

    print("\nCorrect idx distribution:")
    print(g["correct_idx"].value_counts(dropna=False))

    print("\nChoice order distribution:")
    print(g["choices"].apply(tuple).value_counts().head(10))


oos_branch_object_camera_relative_position
N: 100

Target text distribution:
target_text
Front-left     30
Back-left      26
Back-right     25
Front-right    19
Name: count, dtype: int64

Correct idx distribution:
correct_idx
1.0    32
3.0    30
0.0    23
2.0    15
Name: count, dtype: int64

Choice order distribution:
choices
(Front-left, Front-right, Back-left, Back-right)    7
(Front-right, Front-left, Back-left, Back-right)    7
(Back-left, Back-right, Front-right, Front-left)    7
(Front-right, Back-right, Front-left, Back-left)    7
(Front-left, Back-left, Front-right, Back-right)    6
(Back-left, Front-left, Front-right, Back-right)    6
(Back-right, Back-left, Front-left, Front-right)    5
(Back-right, Back-left, Front-right, Front-left)    5
(Front-left, Front-right, Back-right, Back-left)    5
(Back-right, Front-right, Back-left, Front-left)    5
Name: count, dtype: int64

oos_branch_object_object_distance
N: 100

Target text distribution:
target_text
medium    45
far       3

In [87]:
baseline_rows = []

for qclass, g in df.groupby("question_class"):
    n = len(g)

    semantic_counts = g["target_text"].value_counts(dropna=False)
    idx_counts = g["correct_idx"].value_counts(dropna=False)

    semantic_majority_acc = semantic_counts.iloc[0] / n
    idx_majority_acc = idx_counts.iloc[0] / n

    baseline_rows.append({
        "question_class": qclass,
        "n": n,
        "semantic_majority": semantic_counts.index[0],
        "semantic_majority_acc": semantic_majority_acc,
        "idx_majority": idx_counts.index[0],
        "idx_majority_acc": idx_majority_acc,
    })

baseline_df = pd.DataFrame(baseline_rows).sort_values("question_class")
baseline_df




,question_class,n,semantic_majority,semantic_majority_acc,idx_majority,idx_majority_acc
0,oos_branch_object_camera_relative_position,100,Front-left,0.30,1.0,0.32
1,oos_branch_object_object_distance,100,medium,0.45,2.0,0.40
2,oos_branch_object_object_relation,100,Back-left,0.52,3.0,0.30
3,oos_step1_visibility,100,No,1.00,0.0,0.50
4,oos_step2_last_visible,100,"<TIME 00:13:08.0 video 1>; Point=(0.7052, 0.9480)",0.01,NaN,1.00
5,oos_step3_last_placement,100,"<TIME 00:02:17.8 video 1>; Point=(0.7105, 0.6904)",0.11,NaN,1.00
6,oos_step4_fixture,100,counter area beside the hob and close to the door,0.27,1.0,0.28


In [88]:
step1 = df[df["question_class"] == "oos_step1_visibility"].copy()

step1["choices_tuple"] = step1["choices"].apply(tuple)

step1[["choices_tuple", "correct_idx", "target_text"]].value_counts()

choices_tuple  correct_idx  target_text
(No, Yes)      0.0          No             50
(Yes, No)      1.0          No             50
Name: count, dtype: int64

In [89]:
dist = df[df["question_class"] == "oos_branch_object_object_distance"]

print(dist["choices"].apply(len).value_counts())
print(dist["correct_idx"].value_counts(dropna=False))

bad = dist[
    (dist["choices"].apply(len) != 3)
    | (~dist["correct_idx"].isin([0, 1, 2]))
]

bad[["trajectory_id", "step", "target_text", "choices", "correct_idx"]]

choices
3    100
Name: count, dtype: int64
correct_idx
2.0    40
1.0    36
0.0    24
Name: count, dtype: int64


,trajectory_id,step,target_text,choices,correct_idx


In [90]:
import json
from collections import Counter

path = Path("/Users/fangzhouma/Desktop/3d_vision/3D-VLM-benchmark-OOS/outputs/jsonl_v2/multi_turn.jsonl")
rows = []
with open(path, "r") as f:
    for line in f:
        if line.strip():
            rows.append(json.loads(line))

target_names = [r["object_a_name"] for r in rows]
target_ids = [r["object_a_assoc_id"] for r in rows]
query_times = [r["query_time_sec"] for r in rows]

print("Total samples:", len(rows))
print("Unique target object names:", len(set(target_names)))
print("Unique target object IDs:", len(set(target_ids)))

print("Query time range:")
print("min:", min(query_times))
print("max:", max(query_times))

print("\nTarget object counts:")
for name, count in Counter(target_names).most_common():
    print(count, name)

Total samples: 100
Unique target object names: 36
Unique target object IDs: 36
Query time range:
min: 49.0
max: 796.0

Target object counts:
11 both black jars
10 soap dispenser
6 bottle of oil
5 measuring cup
4 bag of lemons
3 washing up liquids bottle
3 can of condensed milk
3 left glove
3 butter's package
3 orange cloths
2 pen
2 cover of food processer bowl
2 empty mug
2 tupperware
2 thermal flask
2 big plastic disk
2 food processor bowl
2 box of hand blender
2 carrot piece
2 packet of biscuits
2 spatula
2 sponge
2 box of stock cubes
2 right glove
2 butter package2
2 thermal flask lid
2 kettle
2 plastic bag
2 bag of flour
2 stock cube
2 box of chicken
2 condiment jar
2 notebook
1 chopping board
1 cooling rack
1 second lemon half
